In [ ]:
## Read new snip_experiments_corrected*.ipynb files and report SNIP pruning results

import nbformat
import re
import numpy as np
import glob

# Regex patterns
initial_parameters = re.compile(r"Initial number of parameters: ([\d.]+)")
total_parameters_after = re.compile(r"parameters after pruning: ([\d.]+)")
total_parameters_testing = re.compile(r"active parameters on testing: ([\d.]+)")
train_val_pattern = re.compile(r"Train Accuracy: ([\d.]+)%, Validation Accuracy: ([\d.]+)%")
test_pattern = re.compile(r"Accuracy on the test set: ([\d.]+)%")
pruning_ratio_pattern = re.compile(r"\s*pruning_ratio=([\d.]+),")
model_pattern = re.compile(r"run_experiment\((\w+)")
dataset_pattern = re.compile(r"torchvision.datasets.(\w+)")

# Read all snip_experiments_corrected*.ipynb notebooks
notebook_files = sorted(glob.glob("snip_experiments_corrected*.ipynb"))

# Store results
all_results = {"MNIST": [], "FashionMNIST": []}

for nb_path in notebook_files:
    if "OLD" in nb_path:
        continue
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    buffer = None

    code_cells = [cell for cell in nb.cells if cell.cell_type == "code"]
    i = 0
    while i < len(code_cells):
        cell = code_cells[i]

        train_text = "".join(o.get("text", "") for o in cell.get("outputs", []) if o["output_type"] == "stream")
        test_text = "".join(o.get("text", "") for o in cell.get("outputs", []) if o["output_type"] == "stream")
        model_code = "".join(cell.get("source", ""))

        dataset_match = dataset_pattern.search(model_code)
        if dataset_match:
            buffer = all_results[dataset_match.group(1)]

        # Check if it matches the expected pattern
        train_val_matches = train_val_pattern.findall(train_text)
        test_match = test_pattern.search(test_text)
        model_match = model_pattern.search(model_code)
        pruning_ratio_match = pruning_ratio_pattern.search(model_code)
        initial_parameters_match = initial_parameters.search(train_text)
        total_parameters_after_match = total_parameters_after.search(train_text)
        total_parameters_testing_match = total_parameters_testing.search(test_text)

        if train_val_matches and test_match and model_match and pruning_ratio_match and initial_parameters_match and total_parameters_after_match and total_parameters_testing_match:
            model = model_match.group(1)
            train_accuracies = [float(t) for t, _ in train_val_matches]
            val_accuracies = [float(v) for _, v in train_val_matches]
            test_accuracy = float(test_match.group(1))
            pruning_ratio = float(pruning_ratio_match.group(1))
            initial_params = int(initial_parameters_match.group(1))
            params_after = int(total_parameters_after_match.group(1))
            params_testing = int(total_parameters_testing_match.group(1))

            buffer.append({
                "model": model,
                "pruning_ratio": pruning_ratio,
                "initial_params": initial_params, 
                "params_after": params_after,
                "params_testing": params_testing,
                "test": test_accuracy
            })

        i += 1

# Group results by model, keeping separate entries for different instances
from collections import defaultdict

for s in ["MNIST", "FashionMNIST"]:
    print(f"{s}:")
    buffer = all_results[s]

    grouped = defaultdict(list)
    for entry in buffer:
        key = entry['model']
        grouped[key].append((entry["initial_params"], entry["params_after"], entry["params_testing"], entry["pruning_ratio"], entry["test"]))

    # Print results with 2 decimal points for arrays
    for key, values in grouped.items():
        values_np = np.stack((values[::4], values[1::4], values[2::4], values[3::4]), axis = 0)
        means = np.mean(values_np, axis=1).T
        stds = np.std(values_np, axis=1).T
        mmax = np.max(values_np, axis=1).T
        mmin = np.min(values_np, axis=1).T

        # Format each element in the arrays to 2 decimal places
        formatted_pruning = ", ".join(f"{x:.4f}" for x in means[3])
        formatted_initial_params = ", ".join(f"{x}" for x in means[0])
        if "MPM" in key:
            formatted_params_after = ", ".join(f"{x:.2f}" for x in mmax[1])
            formatted_params_testing = ", ".join(f"{x:.2f}" for x in mmax[2])
        else:
            formatted_params_after = ", ".join(f"{x:.2f}" for x in mmin[1])
            formatted_params_testing = ", ".join(f"{x:.2f}" for x in mmin[2])
        formatted_compression_mean = ", ".join(f"{x:.2f}" for x in means[4])
        formatted_compression_std = ", ".join(f"{x:.2f}" for x in stds[4])

        print(f"{key}: pruning ratios: [{formatted_pruning}], "
            f"initial params: [{formatted_initial_params}], "
            f"params after pruning: [{formatted_params_after}], " 
            f"params on testing: [{formatted_params_testing}], " 
            f"compression accuracy: [{formatted_compression_mean}] ± [{formatted_compression_std}]")


MNIST:
MLP: pruning ratios: [0.9875, 0.9900, 0.9925, 0.9950], initial params: [466698.0, 466698.0, 466698.0, 466698.0], params after pruning: [5841.00, 4674.00, 3508.00, 2342.00], params on testing: [5841.00, 4674.00, 3508.00, 2342.00], compression accuracy: [95.41, 94.98, 93.97, 88.41] ± [0.24, 0.16, 0.42, 4.99]
RMPM: pruning ratios: [0.9875, 0.9900, 0.9925, 0.9950], initial params: [469268.0, 469268.0, 469268.0, 469268.0], params after pruning: [5895.00, 4722.00, 3549.00, 2376.00], params on testing: [5895.00, 4722.00, 3549.00, 2376.00], compression accuracy: [94.61, 94.22, 93.26, 90.17] ± [0.16, 0.07, 0.22, 0.27]
FashionMNIST:
MLP: pruning ratios: [0.9875, 0.9900, 0.9925, 0.9950], initial params: [466698.0, 466698.0, 466698.0, 466698.0], params after pruning: [5839.00, 4672.00, 3506.00, 2340.00], params on testing: [5839.00, 4672.00, 3506.00, 2340.00], compression accuracy: [85.05, 84.32, 82.92, 79.09] ± [0.06, 0.13, 0.48, 1.33]
RMPM: pruning ratios: [0.9875, 0.9900, 0.9925, 0.9950]

In [ ]:
## Read new snip_resnet_experiments_corrected*.ipynb files and report SNIP pruning results

import nbformat
import re
import numpy as np
import glob

# Regex patterns
initial_parameters = re.compile(r"Initial number of parameters: ([\d.]+)")
total_parameters_after = re.compile(r"parameters after pruning: ([\d.]+)")
total_parameters_testing = re.compile(r"active parameters on testing: ([\d.]+)")
train_val_pattern = re.compile(r"Train Accuracy: ([\d.]+)%, Validation Accuracy: ([\d.]+)%")
test_pattern = re.compile(r"Accuracy on the test set: ([\d.]+)%")
pruning_ratio_pattern = re.compile(r"\s*pruning_ratio=([\d.]+),")
model_pattern = re.compile(r"run_experiment\(\[(\w+), \{\"in_channels\": ([\d.]+)")
method_pattern = re.compile(r"               method=\"(\w+)\"")
dataset_pattern = re.compile(r"torchvision.datasets.(\w+)")

# Read all snip_resnet_experiments_corrected*.ipynb notebooks
notebook_files = sorted(glob.glob("snip_resnet_experiments_corrected*.ipynb"))

# Store results
all_results = {"FashionMNIST": [], "CIFAR10": []}

for nb_path in notebook_files:
    if "OLD" in nb_path:
        continue
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    buffer = None

    code_cells = [cell for cell in nb.cells if cell.cell_type == "code"]
    i = 0
    while i < len(code_cells):
        cell = code_cells[i]

        train_text = "".join(o.get("text", "") for o in cell.get("outputs", []) if o["output_type"] == "stream")
        test_text = "".join(o.get("text", "") for o in cell.get("outputs", []) if o["output_type"] == "stream")
        model_code = "".join(cell.get("source", ""))

        dataset_match = dataset_pattern.search(model_code)
        if dataset_match:
            buffer = all_results[dataset_match.group(1)]

        # Check if it matches the expected pattern
        train_val_matches = train_val_pattern.findall(train_text)
        test_match = test_pattern.search(test_text)
        model_match = model_pattern.search(model_code)
        method_match = method_pattern.search(model_code)
        pruning_ratio_match = pruning_ratio_pattern.search(model_code)
        initial_parameters_match = initial_parameters.search(train_text)
        total_parameters_after_match = total_parameters_after.search(train_text)
        total_parameters_testing_match = total_parameters_testing.search(test_text)

        if train_val_matches and test_match and model_match and method_match and pruning_ratio_match and initial_parameters_match and total_parameters_after_match and total_parameters_testing_match:
            model = f"{model_match.group(1)}(in_channels={model_match.group(2)}, method={method_match.group(1)})"
            train_accuracies = [float(t) for t, _ in train_val_matches]
            val_accuracies = [float(v) for _, v in train_val_matches]
            test_accuracy = float(test_match.group(1))
            pruning_ratio = float(pruning_ratio_match.group(1))
            initial_params = int(initial_parameters_match.group(1))
            params_after = float(total_parameters_after_match.group(1))
            params_testing = float(total_parameters_testing_match.group(1))

            buffer.append({
                "model": model,
                "pruning_ratio": pruning_ratio,
                "initial_params": initial_params, 
                "params_after": params_after,
                "params_testing": params_testing,
                "test": test_accuracy
            })

        i += 1

# Group results by model, keeping separate entries for different instances
from collections import defaultdict

for s in ["FashionMNIST", "CIFAR10"]:
    print(f"{s}:")
    buffer = all_results[s]

    grouped = defaultdict(list)
    for entry in buffer:
        key = entry['model']
        grouped[key].append((entry["initial_params"], entry["params_after"], entry["params_testing"], entry["pruning_ratio"], entry["test"]))

    # Print results with 2 decimal points for arrays
    for key, values in grouped.items(): 
        ## snip = all params treated the same, snip2 = we never prune the linear activations (more important)   
        if "MPM" in key and s == "CIFAR10" and not "snip2" in key:
            continue
        values_np = np.stack((values[::2], values[1::2]), axis = 0)
        means = np.mean(values_np, axis=1).T
        stds = np.std(values_np, axis=1).T
        mmax = np.max(values_np, axis=1).T
        mmin = np.min(values_np, axis=1).T

        # Format each element in the arrays to 2 decimal places
        formatted_pruning = ", ".join(f"{x:.4f}" for x in means[3])
        formatted_initial_params = ", ".join(f"{x}" for x in means[0])
        if "MPM" in key:
            formatted_params_after = ", ".join(f"{x:.2f}" for x in mmax[1])
            formatted_params_testing = ", ".join(f"{x:.2f}" for x in mmax[2])
        else:
            formatted_params_after = ", ".join(f"{x:.2f}" for x in mmin[1])
            formatted_params_testing = ", ".join(f"{x:.2f}" for x in mmin[2])
        formatted_compression_mean = ", ".join(f"{x:.2f}" for x in means[4])
        formatted_compression_std = ", ".join(f"{x:.2f}" for x in stds[4])

        print(f"{key}: pruning ratios: [{formatted_pruning}], "
            f"initial params: [{formatted_initial_params}], "
            f"params after pruning: [{formatted_params_after}], " 
            f"params on testing: [{formatted_params_testing}], " 
            f"compression accuracy: [{formatted_compression_mean}] ± [{formatted_compression_std}]")


FashionMNIST:
ResNet20(in_channels=1, method=snip): pruning ratios: [0.9500, 0.9750], initial params: [271994.0, 271994.0], params after pruning: [14217.00, 7433.00], params on testing: [14217.00, 7433.00], compression accuracy: [91.34, 90.02] ± [0.23, 0.38]
MPM_ResNet20(in_channels=1, method=snip): pruning ratios: [0.9500, 0.9750], initial params: [276810.0, 276810.0], params after pruning: [14458.00, 7554.00], params on testing: [14458.00, 7554.00], compression accuracy: [89.04, 87.26] ± [0.53, 0.26]
MPM_ResNet20(in_channels=1, method=snip2): pruning ratios: [0.9500, 0.9750], initial params: [276810.0, 276810.0], params after pruning: [14458.00, 7554.00], params on testing: [14458.00, 7554.00], compression accuracy: [89.27, 87.76] ± [0.11, 0.22]
CIFAR10:
ResNet20(in_channels=3, method=snip): pruning ratios: [0.9500, 0.9750], initial params: [272282.0, 272282.0], params after pruning: [14231.00, 7440.00], params on testing: [14231.00, 7440.00], compression accuracy: [72.27, 66.78] ± [